# 🧪 Pruebas de Software — Sesiones 6 y 7
## Regression Testing · Smoke Testing · Acceptance Testing

---

**Universidad:** Universitaria de Colombia  
**Materia:** Pruebas de Software  
**Semestre:** 7mo — Ingeniería de Software  
**Docente:** Mg. Sergio Alejandro Burbano Mena  

---

### 📌 ¿Qué aprenderás en este notebook?

| # | Tema | Tiempo estimado |
|---|------|-----------------|
| 1 | Configuración del entorno | 5 min |
| 2 | Sistema de ejemplo: Biblioteca Universitaria | 10 min |
| 3 | **Regression Testing** — Pruebas de Regresión | 25 min |
| 4 | **Smoke Testing** — Pruebas de Humo | 20 min |
| 5 | **Acceptance Testing** — Pruebas de Aceptación (BDD/Gherkin) | 25 min |
| 6 | Integración: los 3 tipos en un pipeline | 10 min |

---

> 💡 **Cómo usar este notebook:**  
> Ejecuta las celdas **en orden, de arriba hacia abajo**.  
> Lee cada comentario antes de ejecutar — el código está diseñado para entenderse leyéndolo.

---
# ⚙️ SECCIÓN 0 — Configuración del Entorno

Instalamos las librerías necesarias para ejecutar los tres tipos de prueba en Colab.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 0.1 — Instalación de dependencias
# ═══════════════════════════════════════════════════════════════════

# 'pytest'      : el framework de pruebas más popular de Python
# 'behave'      : framework para BDD (Behavior Driven Development)
#                 permite escribir pruebas de aceptación en Gherkin
# 'pytest-mock' : permite crear objetos falsos (mocks) en los tests
# 'requests'    : para simular llamadas HTTP en el smoke testing
# 'ipytest'     : permite ejecutar pytest directamente en Colab

!pip install pytest behave pytest-mock requests ipytest --quiet

# Mostramos un mensaje de confirmación cuando todo esté instalado
print("✅ Todas las dependencias instaladas correctamente.")
print("   pytest     → framework de pruebas")
print("   behave     → BDD / Gherkin para acceptance tests")
print("   pytest-mock → mocks y stubs")
print("   ipytest    → ejecutar pytest en Colab")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 0.2 — Configuración de ipytest para ejecutar en Colab
# ═══════════════════════════════════════════════════════════════════

# ipytest nos permite usar la magia %%ipytest en las celdas
# para ejecutar tests de pytest directamente en el notebook
import ipytest

# autoconfig() configura ipytest para que funcione automáticamente
# con el nombre del módulo actual del notebook
ipytest.autoconfig()

# Importamos las librerías estándar que usaremos a lo largo del notebook
import pytest          # framework principal de testing
import datetime        # para manejar fechas en los ejemplos
from unittest.mock import MagicMock, patch  # para crear objetos falsos
from dataclasses import dataclass, field    # para crear clases de datos limpias
from typing import List, Optional           # para type hints

print("✅ Configuración lista. Puedes empezar el notebook.")

---
# 🏛️ SECCIÓN 1 — Sistema de Ejemplo: Biblioteca Universitaria

Para hacer los ejemplos **concretos y realistas**, trabajamos con un sistema que todos conocen.  
Este sistema de biblioteca será el que probaremos con los tres tipos de prueba.

```
📚 Sistema de Biblioteca
├── Módulo de Libros      → consultar disponibilidad, listar catálogo
├── Módulo de Préstamos   → registrar préstamo, registrar devolución
├── Módulo de Multas      → calcular multa, verificar bloqueo
└── Módulo de Usuarios    → autenticar, verificar estado
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 1.1 — Modelos de datos del sistema de biblioteca
# ═══════════════════════════════════════════════════════════════════

# @dataclass es un decorador de Python que genera automáticamente
# __init__, __repr__ y otros métodos a partir de los campos definidos

@dataclass
class Libro:
    """
    Representa un libro del catálogo de la biblioteca.
    
    Atributos:
        id (int)         : identificador único del libro
        titulo (str)     : título del libro
        autor (str)      : nombre del autor
        disponible (bool): True si el libro no está prestado actualmente
    """
    id: int                    # identificador único
    titulo: str                # título del libro
    autor: str                 # nombre del autor
    disponible: bool = True    # por defecto los libros están disponibles


@dataclass
class Estudiante:
    """
    Representa un estudiante de la universidad.
    
    Atributos:
        codigo (str)       : código universitario del estudiante
        nombre (str)       : nombre completo
        activo (bool)      : True si puede hacer nuevos préstamos
        multa_pendiente(float): monto de multa que debe pagar (en pesos COP)
    """
    codigo: str
    nombre: str
    activo: bool = True          # por defecto los estudiantes están activos
    multa_pendiente: float = 0.0 # sin multa al inicio


@dataclass
class Prestamo:
    """
    Representa un préstamo activo de un libro a un estudiante.
    
    Atributos:
        id (int)                  : identificador único del préstamo
        libro_id (int)            : ID del libro prestado
        estudiante_codigo (str)   : código del estudiante que tomó el préstamo
        fecha_prestamo (date)     : cuándo se realizó el préstamo
        fecha_limite (date)       : fecha máxima de devolución (préstamo + 14 días)
        devuelto (bool)           : True si el libro ya fue devuelto
    """
    id: int
    libro_id: int
    estudiante_codigo: str
    fecha_prestamo: datetime.date
    fecha_limite: datetime.date    # 14 días después del préstamo
    devuelto: bool = False         # inicialmente no está devuelto


# Mostramos la estructura de los modelos para que quede clara
print("📚 Modelos creados:")
print()
print("Libro: id, titulo, autor, disponible")
print("Estudiante: codigo, nombre, activo, multa_pendiente")
print("Prestamo: id, libro_id, estudiante_codigo, fecha_prestamo, fecha_limite, devuelto")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 1.2 — Lógica de negocio del sistema de biblioteca
#
#  Esta es la aplicación que vamos a PROBAR.
#  En un proyecto real, este código estaría en archivos separados.
#  Aquí lo ponemos todo junto para que sea fácil de ver.
# ═══════════════════════════════════════════════════════════════════

# ── CONSTANTES DE CONFIGURACIÓN ─────────────────────────────────

DIAS_PRESTAMO = 14         # cuántos días dura un préstamo
TARIFA_DIARIA_MULTA = 500  # pesos COP por día de retraso
LIMITE_PRESTAMOS_ACTIVOS = 3  # máximo de libros simultáneos por estudiante


# ── MÓDULO DE LIBROS ────────────────────────────────────────────

def esta_disponible(libro: Libro) -> bool:
    """
    Verifica si un libro está disponible para préstamo.
    
    Args:
        libro: el objeto Libro a consultar
    
    Returns:
        True si el libro está disponible, False si está prestado
    """
    # Simplemente retornamos el atributo 'disponible' del libro
    # En una app real consultaría la base de datos
    return libro.disponible


def buscar_libro_por_titulo(catalogo: List[Libro], titulo: str) -> Optional[Libro]:
    """
    Busca un libro en el catálogo por su título (búsqueda parcial).
    
    Args:
        catalogo: lista de todos los libros disponibles
        titulo:   texto a buscar en el título
    
    Returns:
        El primer Libro cuyo título contenga el texto buscado,
        o None si no se encuentra ninguno.
    """
    # Recorremos todos los libros del catálogo
    for libro in catalogo:
        # .lower() convierte a minúsculas para hacer la búsqueda
        # sin importar mayúsculas o minúsculas
        if titulo.lower() in libro.titulo.lower():
            return libro   # retornamos el primero que coincida
    
    # Si no encontramos ninguno, retornamos None
    return None


# ── MÓDULO DE PRÉSTAMOS ─────────────────────────────────────────

def registrar_prestamo(
    libro: Libro,
    estudiante: Estudiante,
    prestamos_activos: List[Prestamo],
    ultimo_id: int = 1
) -> Prestamo:
    """
    Registra un nuevo préstamo de un libro a un estudiante.
    
    Reglas de negocio:
    - El libro debe estar disponible
    - El estudiante debe estar activo (sin multas bloqueantes)
    - El estudiante no puede tener más de LIMITE_PRESTAMOS_ACTIVOS préstamos
    
    Args:
        libro:            el libro que se quiere prestar
        estudiante:       el estudiante que lo solicita
        prestamos_activos:lista de préstamos actuales del estudiante
        ultimo_id:        número para generar el ID del nuevo préstamo
    
    Returns:
        El objeto Prestamo creado
    
    Raises:
        ValueError: si el libro no está disponible
        ValueError: si el estudiante no está activo
        ValueError: si el estudiante ya tiene el máximo de préstamos
    """
    # VALIDACIÓN 1: el libro debe estar disponible
    if not libro.disponible:
        # Lanzamos un error descriptivo que explica exactamente qué falló
        raise ValueError(f"El libro '{libro.titulo}' no está disponible")
    
    # VALIDACIÓN 2: el estudiante debe poder hacer préstamos
    if not estudiante.activo:
        raise ValueError(
            f"El estudiante '{estudiante.nombre}' tiene multa pendiente "
            f"de ${estudiante.multa_pendiente:,.0f} COP. "
            "Debe pagarla antes de hacer nuevos préstamos."
        )
    
    # VALIDACIÓN 3: el estudiante no puede tener demasiados préstamos
    # Contamos solo los préstamos que NO han sido devueltos
    prestamos_sin_devolver = [p for p in prestamos_activos if not p.devuelto]
    if len(prestamos_sin_devolver) >= LIMITE_PRESTAMOS_ACTIVOS:
        raise ValueError(
            f"El estudiante ya tiene {LIMITE_PRESTAMOS_ACTIVOS} préstamos activos. "
            "Debe devolver uno antes de solicitar otro."
        )
    
    # Si pasó todas las validaciones, creamos el préstamo
    hoy = datetime.date.today()                    # fecha actual
    limite = hoy + datetime.timedelta(days=DIAS_PRESTAMO)  # 14 días después
    
    # Creamos el objeto Prestamo con todos sus datos
    nuevo_prestamo = Prestamo(
        id=ultimo_id,
        libro_id=libro.id,
        estudiante_codigo=estudiante.codigo,
        fecha_prestamo=hoy,
        fecha_limite=limite
    )
    
    # Actualizamos el estado del libro a "no disponible"
    libro.disponible = False
    
    return nuevo_prestamo


def registrar_devolucion(
    prestamo: Prestamo,
    libro: Libro,
    fecha_devolucion: Optional[datetime.date] = None
) -> float:
    """
    Registra la devolución de un libro y calcula la multa si aplica.
    
    Args:
        prestamo:         el préstamo que se va a cerrar
        libro:            el libro que se devuelve
        fecha_devolucion: fecha en que se devuelve (default: hoy)
    
    Returns:
        El monto de la multa en pesos COP (0.0 si fue a tiempo)
    
    Raises:
        ValueError: si el préstamo ya fue registrado como devuelto
    """
    # Verificamos que el préstamo no haya sido devuelto ya
    if prestamo.devuelto:
        raise ValueError(f"El préstamo #{prestamo.id} ya fue registrado como devuelto")
    
    # Si no se especificó fecha, usamos hoy
    if fecha_devolucion is None:
        fecha_devolucion = datetime.date.today()
    
    # Calculamos cuántos días de retraso hubo
    # timedelta puede ser negativo si devolvió antes de la fecha límite
    diferencia = fecha_devolucion - prestamo.fecha_limite
    dias_retraso = diferencia.days  # días de diferencia (negativo = antes de tiempo)
    
    # La multa solo aplica si hay días de retraso (dias_retraso > 0)
    # max(0, dias_retraso) asegura que no haya multas negativas
    multa = max(0, dias_retraso) * TARIFA_DIARIA_MULTA
    
    # Actualizamos el estado del préstamo
    prestamo.devuelto = True
    
    # El libro vuelve a estar disponible para otros estudiantes
    libro.disponible = True
    
    return multa


# ── MÓDULO DE MULTAS ────────────────────────────────────────────

def calcular_multa_por_dias(dias_retraso: int) -> float:
    """
    Calcula el monto de la multa dado un número de días de retraso.
    
    Fórmula: multa = días_retraso × TARIFA_DIARIA_MULTA
    
    Args:
        dias_retraso: número de días que pasaron después de la fecha límite
    
    Returns:
        Monto de la multa en pesos COP
    
    Raises:
        ValueError: si los días de retraso son negativos (no tiene sentido)
    """
    # Los días de retraso nunca pueden ser negativos
    if dias_retraso < 0:
        raise ValueError(f"Los días de retraso no pueden ser negativos: {dias_retraso}")
    
    # Multiplicamos los días por la tarifa configurada
    return dias_retraso * TARIFA_DIARIA_MULTA


def aplicar_multa_a_estudiante(estudiante: Estudiante, monto: float) -> None:
    """
    Aplica una multa a un estudiante y actualiza su estado.
    Si la multa es mayor a $0, el estudiante queda BLOQUEADO.
    
    Args:
        estudiante: el objeto Estudiante a quien se aplica la multa
        monto:      el monto de la multa en pesos COP
    """
    # Sumamos la multa nueva a cualquier multa pendiente anterior
    estudiante.multa_pendiente += monto
    
    # Si tiene cualquier multa pendiente, bloqueamos su cuenta
    # El estudiante no podrá hacer nuevos préstamos hasta pagar
    if estudiante.multa_pendiente > 0:
        estudiante.activo = False


def pagar_multa(estudiante: Estudiante, monto_pago: float) -> float:
    """
    Registra un pago de multa y desbloquea al estudiante si paga todo.
    
    Args:
        estudiante:   el estudiante que paga
        monto_pago:   cuánto está pagando
    
    Returns:
        El saldo pendiente después del pago
    
    Raises:
        ValueError: si el monto de pago es negativo o cero
    """
    # Validar que el pago sea un monto positivo
    if monto_pago <= 0:
        raise ValueError("El monto de pago debe ser mayor a cero")
    
    # Restamos el pago de la multa pendiente
    # max(0, ...) evita que la multa quede en negativo
    estudiante.multa_pendiente = max(0.0, estudiante.multa_pendiente - monto_pago)
    
    # Si ya no tiene multa, reactivamos su cuenta
    if estudiante.multa_pendiente == 0.0:
        estudiante.activo = True
    
    return estudiante.multa_pendiente


# ── MÓDULO DE AUTENTICACIÓN (simulado) ─────────────────────────

def autenticar_usuario(codigo: str, contrasena: str) -> Optional[Estudiante]:
    """
    Simula la autenticación de un usuario.
    En una app real, consultaría la base de datos.
    Aquí solo verificamos credenciales hardcodeadas para los ejemplos.
    
    Args:
        codigo:     código universitario del estudiante
        contrasena: contraseña del estudiante
    
    Returns:
        El objeto Estudiante si las credenciales son correctas,
        None si son incorrectas.
    """
    # Base de datos simulada de usuarios
    usuarios_validos = {
        "2024-001": ("pass123", Estudiante("2024-001", "Ana García")),
        "2024-002": ("clave456", Estudiante("2024-002", "Carlos López")),
        "2024-003": ("test789", Estudiante("2024-003", "María Rodríguez")),
    }
    
    # Buscamos el usuario en nuestra BD simulada
    if codigo in usuarios_validos:
        contrasena_correcta, estudiante = usuarios_validos[codigo]
        # Solo retornamos el estudiante si la contraseña coincide
        if contrasena == contrasena_correcta:
            return estudiante
    
    # Credenciales incorrectas → retornamos None
    return None


print("✅ Sistema de Biblioteca cargado correctamente.")
print()
print("📦 Funciones disponibles:")
print("   LIBROS   : esta_disponible(), buscar_libro_por_titulo()")
print("   PRÉSTAMOS: registrar_prestamo(), registrar_devolucion()")
print("   MULTAS   : calcular_multa_por_dias(), aplicar_multa_a_estudiante(), pagar_multa()")
print("   AUTH     : autenticar_usuario()")

---
# 🔁 SECCIÓN 2 — REGRESSION TESTING (Pruebas de Regresión)

## ¿Qué es Regression Testing?

Las **pruebas de regresión** verifican que los cambios recientes en el código  
**no rompieron funcionalidades que ya funcionaban correctamente**.

### Analogía simple:
> Imagina que tienes un carro que funciona bien. Le cambias el sistema de frenos.  
> Después del cambio, pruebas que los frenos funcionen (prueba del nuevo cambio).  
> Pero **también pruebas que el motor, las luces y la dirección sigan funcionando** — eso es regression testing.

### ¿Cuándo se ejecutan?
```
Desarrollador hace un cambio
        ↓
Se ejecutan los regression tests
        ↓
¿Algún test falló?  →  SÍ  →  El cambio ROMPIÓ algo → Fix requerido
        ↓
        NO  →  El cambio es seguro → Continuar
```

### Tipos de regression testing:
| Tipo | Descripción | Cuándo usarlo |
|------|-------------|---------------|
| **Completa** | Ejecuta TODOS los tests | Releases mayores |
| **Parcial** | Solo el área afectada | Cambios pequeños |
| **Selectiva** | Por análisis de impacto | Cambios transversales |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 2.1 — Demostración: Por qué ocurren las regresiones
#
#  Escenario: El desarrollador modificó calcular_multa_por_dias()
#  para agregar un descuento del 10% en multas mayores a $5000.
#  Sin querer, rompió el cálculo básico.
# ═══════════════════════════════════════════════════════════════════

print("="*60)
print(" DEMOSTRACIÓN: Cómo ocurre una regresión")
print("="*60)

# ── VERSIÓN ORIGINAL (funciona bien) ────────────────────────────
def calcular_multa_v1(dias_retraso: int) -> float:
    """
    VERSIÓN 1 — Original.
    Calcula la multa multiplicando días por tarifa diaria.
    Esta versión funciona correctamente.
    """
    if dias_retraso < 0:
        raise ValueError("Los días de retraso no pueden ser negativos")
    return dias_retraso * TARIFA_DIARIA_MULTA  # días × $500


# ── VERSIÓN CON REGRESIÓN (tiene un bug) ────────────────────────
def calcular_multa_v2_con_bug(dias_retraso: int) -> float:
    """
    VERSIÓN 2 — Con un bug introducido al agregar el descuento.
    El desarrollador quiso agregar: si multa > 5000, dar 10% de descuento.
    Pero al refactorizar, olvidó incluir el caso base (sin descuento).
    """
    if dias_retraso < 0:
        raise ValueError("Los días de retraso no pueden ser negativos")
    
    multa = dias_retraso * TARIFA_DIARIA_MULTA
    
    # BUG: esta condición aplica el descuento incluso cuando multa <= 5000
    # porque el 'else' del if-else original fue eliminado por error
    if multa > 5000:
        return multa * 0.90  # 10% de descuento
    # ERROR: no hay 'else: return multa' aquí
    # Python retorna None implícitamente cuando no hay return
    # Esto causará errores al intentar hacer cálculos con el resultado


# Comparamos el comportamiento de ambas versiones
casos_prueba = [1, 3, 5, 10, 15]

print(f"\n{'Días':<8} {'V1 (original)':<20} {'V2 (con bug)':<20} {'¿Iguales?':<12}")
print("-" * 60)

for dias in casos_prueba:
    resultado_v1 = calcular_multa_v1(dias)
    resultado_v2 = calcular_multa_v2_con_bug(dias)
    
    # Comparamos los resultados
    son_iguales = resultado_v1 == resultado_v2
    # El símbolo ✅ indica que no hay regresión, ❌ indica una regresión detectada
    estado = "✅ OK" if son_iguales else "❌ REGRESIÓN"
    
    print(f"{dias:<8} ${resultado_v1:<19,.0f} ${str(resultado_v2):<19} {estado}")

print()
print("👆 Nota: para días ≤ 10 (multa ≤ $5000), la V2 retorna None en vez del monto")
print("   Eso es exactamente lo que los regression tests van a detectar.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 2.2 — Suite de Regression Tests para el módulo de multas
#
#  Estos tests se escriben UNA VEZ y se ejecutan después de CADA
#  cambio al código. Si alguno falla, hay una regresión.
# ═══════════════════════════════════════════════════════════════════

%%ipytest -v

# ── La magia %%ipytest le dice a Colab que ejecute pytest en esta celda ──
# -v significa 'verbose': muestra el nombre de cada test y si pasó o falló

import pytest  # importamos pytest dentro de la celda del test

# ═══════════════════════════════════════════════════════════════════
# CLASE DE REGRESIÓN: TestMultasRegression
#
# Convención de nombres en pytest:
# - Las clases de test deben empezar con 'Test'
# - Los métodos de test deben empezar con 'test_'
# - El nombre debe describir QUÉ se está probando
# ═══════════════════════════════════════════════════════════════════

class TestMultasRegression:
    """
    Suite de regression tests para el módulo de multas.
    
    PROPÓSITO: Verificar que cualquier cambio al módulo de multas
    no rompe el comportamiento ya validado y aprobado.
    
    Estos tests deben ejecutarse después de CADA modificación
    al módulo de cálculo de multas.
    """

    # ── Test 1: Caso base ────────────────────────────────────────
    def test_multa_cero_por_devolucion_a_tiempo(self):
        """
        REGRESIÓN: Verificar que devolver a tiempo NO genera multa.
        
        Si este test falla después de un cambio, el cambio rompió
        el caso más básico: sin retraso = sin multa.
        """
        # ARRANGE (preparar): 0 días de retraso = entregó exactamente a tiempo
        dias_retraso = 0
        
        # ACT (actuar): calculamos la multa
        multa = calcular_multa_por_dias(dias_retraso)
        
        # ASSERT (verificar): la multa debe ser exactamente 0
        assert multa == 0.0, (
            f"Se esperaba multa de $0, pero se obtuvo ${multa}. "
            "La devolución a tiempo nunca debe generar multa."
        )

    # ── Test 2: Un día de retraso ────────────────────────────────
    def test_multa_un_dia_retraso(self):
        """
        REGRESIÓN: 1 día de retraso = 1 × $500 = $500.
        Verifica la fórmula más simple.
        """
        # ARRANGE
        dias_retraso = 1
        multa_esperada = 500.0  # 1 día × $500 por día
        
        # ACT
        multa_calculada = calcular_multa_por_dias(dias_retraso)
        
        # ASSERT
        assert multa_calculada == multa_esperada, (
            f"1 día de retraso debería generar ${multa_esperada}, "
            f"pero se calculó ${multa_calculada}"
        )

    # ── Test 3: Tres días de retraso (caso más común) ────────────
    def test_multa_tres_dias_retraso(self):
        """
        REGRESIÓN: 3 días de retraso = 3 × $500 = $1.500.
        Este es el caso que más comúnmente reportan los usuarios.
        """
        dias_retraso = 3
        multa_esperada = 1500.0  # 3 días × $500 por día
        
        multa_calculada = calcular_multa_por_dias(dias_retraso)
        
        assert multa_calculada == multa_esperada

    # ── Test 4: Diez días de retraso ─────────────────────────────
    def test_multa_diez_dias_retraso(self):
        """
        REGRESIÓN: 10 días de retraso = 10 × $500 = $5.000.
        Valor límite justo antes del descuento hipotético del 10%.
        """
        dias_retraso = 10
        multa_esperada = 5000.0  # 10 × $500
        
        multa_calculada = calcular_multa_por_dias(dias_retraso)
        
        assert multa_calculada == multa_esperada

    # ── Test 5: El tipo de retorno debe ser numérico ─────────────
    def test_multa_retorna_float_no_none(self):
        """
        REGRESIÓN: La función debe retornar siempre un número,
        NUNCA None. Este test protege contra el bug de la v2
        que demostrado arriba donde faltó el 'return'.
        """
        # Probamos con múltiples valores para cubrir diferentes ramas del código
        casos = [0, 1, 5, 10, 15, 20]
        
        for dias in casos:
            resultado = calcular_multa_por_dias(dias)
            
            # Verificamos que el resultado NO sea None
            assert resultado is not None, (
                f"La función retornó None para {dias} días de retraso. "
                "Debe retornar siempre un número."
            )
            
            # Verificamos que sea un número (int o float)
            assert isinstance(resultado, (int, float)), (
                f"Se esperaba un número pero se obtuvo {type(resultado)}"
            )

    # ── Test 6: Días negativos deben lanzar error ────────────────
    def test_dias_negativos_lanza_error(self):
        """
        REGRESIÓN: Los días de retraso no pueden ser negativos.
        Esta validación existe desde la versión 1. Cualquier
        refactor debe mantenerla.
        """
        # pytest.raises() verifica que se lance la excepción esperada
        # Si la función NO lanza ValueError, este test FALLA
        with pytest.raises(ValueError) as info_error:
            calcular_multa_por_dias(-5)  # días negativos = error
        
        # También podemos verificar el mensaje del error
        # str(info_error.value) contiene el mensaje de la excepción
        assert "negativo" in str(info_error.value).lower(), (
            "El mensaje de error debe mencionar 'negativo'"
        )

    # ── Test 7: La multa es proporcional (no lineal rota) ────────
    def test_multa_es_proporcional_a_los_dias(self):
        """
        REGRESIÓN: La multa debe ser PROPORCIONAL a los días.
        El doble de días = el doble de multa.
        Este test detecta si alguien introduce un descuento o
        un cargo fijo que rompe la proporcionalidad.
        """
        multa_5_dias  = calcular_multa_por_dias(5)
        multa_10_dias = calcular_multa_por_dias(10)
        
        # 10 días debe costar exactamente el doble que 5 días
        assert multa_10_dias == multa_5_dias * 2, (
            f"La multa de 10 días (${multa_10_dias}) debería ser "
            f"el doble de la de 5 días (${multa_5_dias}). "
            "La relación debe ser proporcional."
        )

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 2.3 — Regression Tests para el módulo de préstamos
#
#  Estos tests protegen las reglas de negocio del préstamo.
#  Si alguien modifica registrar_prestamo(), estos tests
#  deben seguir pasando.
# ═══════════════════════════════════════════════════════════════════

%%ipytest -v

class TestPrestamosRegression:
    """
    Suite de regression tests para el módulo de préstamos.
    Protege las reglas de negocio que YA FUERON validadas y aprobadas.
    """

    # ── Fixture: datos reutilizables ─────────────────────────────
    # Un fixture es una función que prepara datos para los tests.
    # @pytest.fixture crea objetos nuevos para CADA test,
    # evitando que un test afecte al siguiente.
    
    def setup_method(self):
        """
        setup_method() se ejecuta ANTES de cada test de esta clase.
        Crea objetos frescos para que cada test sea independiente.
        """
        # Creamos un libro disponible para prestar
        self.libro = Libro(
            id=1,
            titulo="Clean Code",
            autor="Robert C. Martin"
        )  # disponible=True por defecto
        
        # Creamos un estudiante activo (sin multas)
        self.estudiante = Estudiante(
            codigo="2024-001",
            nombre="Ana García"
        )  # activo=True, multa_pendiente=0.0 por defecto
        
        # Lista vacía de préstamos activos
        self.prestamos_activos = []

    # ── Test 1: Préstamo exitoso ─────────────────────────────────
    def test_prestamo_exitoso_retorna_objeto_prestamo(self):
        """
        REGRESIÓN: Un préstamo válido debe retornar un objeto Prestamo.
        """
        # ACT: registramos el préstamo
        prestamo = registrar_prestamo(
            self.libro,
            self.estudiante,
            self.prestamos_activos
        )
        
        # ASSERT: verificamos que es un objeto Prestamo válido
        assert isinstance(prestamo, Prestamo), "Debe retornar un objeto Prestamo"
        assert prestamo.libro_id == self.libro.id
        assert prestamo.estudiante_codigo == self.estudiante.codigo
        assert prestamo.devuelto == False

    # ── Test 2: Libro queda no disponible ────────────────────────
    def test_prestamo_marca_libro_como_no_disponible(self):
        """
        REGRESIÓN: Al prestar un libro, debe quedar marcado
        como NO disponible. Este comportamiento es crítico:
        sin él, el mismo libro podría prestarse dos veces.
        """
        # Verificamos que el libro ESTÁ disponible antes del préstamo
        assert self.libro.disponible == True, "El libro debe estar disponible inicialmente"
        
        # Realizamos el préstamo
        registrar_prestamo(self.libro, self.estudiante, self.prestamos_activos)
        
        # Verificamos que ahora NO está disponible
        assert self.libro.disponible == False, (
            "Después del préstamo, el libro NO debe estar disponible. "
            "Si este test falla, dos estudiantes podrían tener el mismo libro."
        )

    # ── Test 3: Libro no disponible bloquea el préstamo ──────────
    def test_no_puede_prestar_libro_no_disponible(self):
        """
        REGRESIÓN: No se puede prestar un libro que ya está prestado.
        """
        # Marcamos el libro como no disponible (como si ya estuviera prestado)
        self.libro.disponible = False
        
        # Intentamos prestarlo — debe lanzar ValueError
        with pytest.raises(ValueError) as error_info:
            registrar_prestamo(self.libro, self.estudiante, self.prestamos_activos)
        
        # Verificamos que el mensaje de error sea informativo
        assert "disponible" in str(error_info.value).lower()

    # ── Test 4: Estudiante bloqueado no puede prestar ────────────
    def test_estudiante_con_multa_no_puede_prestar(self):
        """
        REGRESIÓN: Un estudiante con multa pendiente NO puede
        hacer nuevos préstamos. Esta es una regla de negocio crítica.
        """
        # Bloqueamos al estudiante (simula que tiene una multa)
        self.estudiante.activo = False
        self.estudiante.multa_pendiente = 1500.0
        
        with pytest.raises(ValueError) as error_info:
            registrar_prestamo(self.libro, self.estudiante, self.prestamos_activos)
        
        # El mensaje debe mencionar la multa o el bloqueo
        mensaje = str(error_info.value).lower()
        assert "multa" in mensaje or "activo" in mensaje or "bloqueado" in mensaje

    # ── Test 5: Fecha límite = hoy + 14 días ─────────────────────
    def test_fecha_limite_es_14_dias_desde_hoy(self):
        """
        REGRESIÓN: La fecha límite de devolución debe ser
        exactamente 14 días después del préstamo.
        """
        prestamo = registrar_prestamo(self.libro, self.estudiante, self.prestamos_activos)
        
        hoy = datetime.date.today()
        fecha_limite_esperada = hoy + datetime.timedelta(days=DIAS_PRESTAMO)  # 14 días
        
        assert prestamo.fecha_limite == fecha_limite_esperada, (
            f"La fecha límite debe ser {fecha_limite_esperada}, "
            f"pero se calculó {prestamo.fecha_limite}"
        )

    # ── Test 6: Límite de préstamos simultáneos ──────────────────
    def test_no_puede_exceder_limite_prestamos_simultaneos(self):
        """
        REGRESIÓN: Un estudiante no puede tener más de
        LIMITE_PRESTAMOS_ACTIVOS préstamos al mismo tiempo.
        """
        # Creamos préstamos falsos que simulan que el estudiante ya
        # tiene el máximo de préstamos activos
        # devuelto=False significa que aún no los ha devuelto
        prestamos_al_limite = [
            Prestamo(
                id=i+1,
                libro_id=i+10,
                estudiante_codigo=self.estudiante.codigo,
                fecha_prestamo=datetime.date.today(),
                fecha_limite=datetime.date.today() + datetime.timedelta(days=14),
                devuelto=False  # no devuelto = cuenta hacia el límite
            )
            for i in range(LIMITE_PRESTAMOS_ACTIVOS)  # crear exactamente el límite
        ]
        
        # Intentar agregar uno más debe fallar
        with pytest.raises(ValueError) as error_info:
            registrar_prestamo(self.libro, self.estudiante, prestamos_al_limite)
        
        # Verificamos que el mensaje mencione el límite
        assert str(LIMITE_PRESTAMOS_ACTIVOS) in str(error_info.value)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 2.4 — Regression Test del Bug Histórico
#
#  CONTEXTO: En el semestre pasado, existía un bug donde
#  si un estudiante pagaba su multa exactamente con el monto
#  adeudado, su cuenta NO se reactivaba (quedaba bloqueado).
#  Fue reportado, corregido, y este test asegura que NO VUELVA.
# ═══════════════════════════════════════════════════════════════════

%%ipytest -v

class TestBugHistoricoRegression:
    """
    Tests de regresión para bugs históricos documentados.
    
    IMPORTANTE: Estos tests llevan el nombre del bug para que
    sea fácil identificar por qué existen. Nunca los borres
    sin entender por qué fueron creados.
    """

    # ── Bug histórico: pago exacto no reactivaba la cuenta ───────
    def test_bug_pago_exacto_reactiva_cuenta_estudiante(self):
        """
        BUG HISTÓRICO: Al pagar exactamente el monto adeudado,
        la cuenta del estudiante debe quedar ACTIVA.
        
        El bug original: se usaba '<' en vez de '<=' para comparar
        el saldo pendiente, entonces al llegar a 0.0 el sistema
        pensaba que aún debía dinero.
        """
        # ARRANGE: estudiante con multa de $1500
        estudiante = Estudiante(
            codigo="2024-BUG",
            nombre="Estudiante de Prueba",
            activo=False,          # bloqueado por multa
            multa_pendiente=1500.0  # debe $1500
        )
        
        # Verificamos el estado inicial
        assert estudiante.activo == False
        assert estudiante.multa_pendiente == 1500.0
        
        # ACT: paga EXACTAMENTE lo que debe
        saldo_restante = pagar_multa(estudiante, monto_pago=1500.0)
        
        # ASSERT: debe quedar activo y sin deuda
        assert saldo_restante == 0.0, (
            f"Después de pagar $1500 una deuda de $1500, "
            f"el saldo debe ser $0 pero quedó ${saldo_restante}"
        )
        assert estudiante.activo == True, (
            "BUG REGRESÓ: Al pagar exactamente la deuda, "
            "la cuenta debe quedar ACTIVA. Si este test falla, "
            "el bug histórico regresó al código."
        )
        assert estudiante.multa_pendiente == 0.0

    # ── Test adicional: pago parcial no reactiva ─────────────────
    def test_pago_parcial_no_reactiva_cuenta(self):
        """
        REGRESIÓN: Un pago parcial NO debe reactivar la cuenta.
        El estudiante debe pagar TODO para ser desbloqueado.
        """
        # ARRANGE: estudiante con multa de $2000
        estudiante = Estudiante(
            codigo="2024-PAR",
            nombre="Pago Parcial",
            activo=False,
            multa_pendiente=2000.0
        )
        
        # ACT: paga solo $500 (25% de la deuda)
        saldo = pagar_multa(estudiante, monto_pago=500.0)
        
        # ASSERT: debe seguir bloqueado (aún debe $1500)
        assert saldo == 1500.0, f"El saldo debe ser $1500 pero quedó ${saldo}"
        assert estudiante.activo == False, (
            "Con deuda pendiente, el estudiante debe seguir BLOQUEADO"
        )

    # ── Test adicional: verificar que pago de 0 lanza error ──────
    def test_pago_de_cero_o_negativo_lanza_error(self):
        """
        REGRESIÓN: No se puede pagar $0 o un monto negativo.
        """
        estudiante = Estudiante(
            codigo="2024-ERR",
            nombre="Error Pago",
            activo=False,
            multa_pendiente=1000.0
        )
        
        # Intentar pagar $0 debe lanzar ValueError
        with pytest.raises(ValueError):
            pagar_multa(estudiante, monto_pago=0)
        
        # Intentar pagar un monto negativo también debe fallar
        with pytest.raises(ValueError):
            pagar_multa(estudiante, monto_pago=-100)

---
# 💨 SECCIÓN 3 — SMOKE TESTING (Pruebas de Humo)

## ¿Qué es Smoke Testing?

El **Smoke Testing** es un conjunto **pequeño y rápido** de pruebas que verifica que  
las funciones **más críticas** del sistema estén operativas cuando llega un nuevo build.

### Origen del nombre:
> En electrónica, cuando se recibía un circuito nuevo, se conectaba a la corriente.  
> **Si salía humo → el circuito fallaba.** En software: si falla el smoke test → el build se rechaza.

### Reglas del Smoke Testing:
| Regla | Descripción |
|-------|-------------|
| **Rápido** | Debe completarse en 5-15 minutos máximo |
| **Pequeño** | Solo 5-15% de los tests más críticos |
| **Binario** | Pasa todo → continuar. Falla algo → rechazar build |
| **Automático** | Sin intervención humana, corre en CI/CD |

### Diferencia con Regression:
```
Smoke:      ¿EL SISTEMA ESTÁ VIVO?  → amplio pero superficial
Regression: ¿NADA SE ROMPIÓ?        → profundo en áreas afectadas
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 3.1 — Simulador de API REST para el sistema de biblioteca
#
#  En un proyecto real, el smoke test llamaría a endpoints HTTP reales.
#  Aquí simulamos la API con un objeto Python para poder ejecutar
#  los ejemplos sin necesitar un servidor real.
# ═══════════════════════════════════════════════════════════════════

@dataclass
class RespuestaAPI:
    """
    Simula la respuesta de un endpoint HTTP.
    
    En una app real usarías requests.Response,
    que tiene exactamente estos mismos atributos.
    """
    status_code: int   # código HTTP: 200=OK, 404=No encontrado, 500=Error servidor
    data: dict         # el contenido de la respuesta como diccionario
    
    def json(self) -> dict:
        """Simula el método .json() de requests.Response"""
        return self.data


class SimuladorAPIBiblioteca:
    """
    Simula el servidor de la API de la biblioteca.
    
    Podemos configurarla para simular:
    - Una API que funciona correctamente (estado='ok')
    - Una API con la BD caída (estado='db_down')
    - Una API completamente caída (estado='down')
    """
    
    def __init__(self, estado: str = 'ok'):
        """
        Args:
            estado: 'ok' = todo funciona
                    'db_down' = la BD no responde
                    'down' = el servidor está caído
        """
        self.estado = estado
    
    def get(self, endpoint: str) -> RespuestaAPI:
        """Simula una petición GET a un endpoint."""
        
        # Si el servidor está completamente caído, todos los endpoints fallan
        if self.estado == 'down':
            # ConnectionError simula que el servidor no responde
            raise ConnectionError("No se puede conectar al servidor")
        
        # Mapeamos los endpoints a sus respuestas simuladas
        respuestas = {
            '/api/health': self._health(),
            '/api/health/db': self._health_db(),
            '/api/libros': self._listar_libros(),
            '/api/multas/2024-001': self._consultar_multas(),
        }
        
        # Si el endpoint existe, retornamos su respuesta
        if endpoint in respuestas:
            return respuestas[endpoint]
        
        # Endpoint desconocido → 404 Not Found
        return RespuestaAPI(status_code=404, data={'error': 'Endpoint no encontrado'})
    
    def post(self, endpoint: str, data: dict) -> RespuestaAPI:
        """Simula una petición POST."""
        
        if self.estado == 'down':
            raise ConnectionError("No se puede conectar al servidor")
        
        if endpoint == '/api/auth/login':
            return self._login(data)
        
        return RespuestaAPI(status_code=404, data={'error': 'Endpoint no encontrado'})
    
    # ── Respuestas internas según el estado del sistema ──────────
    
    def _health(self) -> RespuestaAPI:
        """Respuesta del endpoint /api/health"""
        if self.estado == 'ok':
            return RespuestaAPI(200, {'status': 'ok', 'version': '2.1.0'})
        elif self.estado == 'db_down':
            # La app responde pero avisa que la BD está caída
            return RespuestaAPI(200, {'status': 'degradado', 'db': 'desconectada'})
        return RespuestaAPI(500, {'status': 'error'})
    
    def _health_db(self) -> RespuestaAPI:
        """Respuesta del endpoint /api/health/db"""
        if self.estado == 'ok':
            return RespuestaAPI(200, {'db_status': 'connected', 'latencia_ms': 12})
        elif self.estado == 'db_down':
            return RespuestaAPI(503, {'db_status': 'disconnected', 'error': 'timeout'})
        return RespuestaAPI(500, {'db_status': 'unknown'})
    
    def _login(self, data: dict) -> RespuestaAPI:
        """Respuesta del endpoint POST /api/auth/login"""
        credenciales_validas = {
            'test@uni.edu.co': 'Test123!',
            '2024-001': 'pass123'
        }
        email = data.get('email', '')
        password = data.get('password', '')
        
        if email in credenciales_validas and credenciales_validas[email] == password:
            return RespuestaAPI(200, {
                'access_token': 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9...',
                'user': {'nombre': 'Estudiante Test', 'codigo': '2024-001'}
            })
        
        return RespuestaAPI(401, {'error': 'Credenciales incorrectas'})
    
    def _listar_libros(self) -> RespuestaAPI:
        """Respuesta del endpoint GET /api/libros"""
        if self.estado == 'db_down':
            return RespuestaAPI(503, {'error': 'Base de datos no disponible'})
        return RespuestaAPI(200, {
            'libros': [
                {'id': 1, 'titulo': 'Clean Code', 'disponible': True},
                {'id': 2, 'titulo': 'The Pragmatic Programmer', 'disponible': False},
                {'id': 3, 'titulo': 'Refactoring', 'disponible': True},
            ],
            'total': 3
        })
    
    def _consultar_multas(self) -> RespuestaAPI:
        """Respuesta del endpoint GET /api/multas/{codigo}"""
        return RespuestaAPI(200, {'multa_total': 0.0, 'prestamos_vencidos': []})


print("✅ Simulador de API creado.")
print("   Modos: SimuladorAPIBiblioteca('ok') | ('db_down') | ('down')")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 3.2 — Suite de Smoke Tests
#
#  Estos son los 5 tests más críticos del sistema.
#  Representan el "checklist mínimo" para saber si el build
#  está listo para pruebas más profundas.
# ═══════════════════════════════════════════════════════════════════

%%ipytest -v

# El marcador @pytest.mark.smoke agrupa estos tests
# Para ejecutar solo smoke tests: pytest -m smoke

class TestBibliotecaSmoke:
    """
    Suite de Smoke Tests para la API de la Biblioteca Universitaria.
    
    PROPÓSITO: Verificar en menos de 5 minutos que el nuevo build
    tiene las funciones básicas operativas.
    
    REGLA: Si CUALQUIERA de estos tests falla → RECHAZAR el build.
    No ejecutar más tests hasta que esto esté resuelto.
    """

    def setup_method(self):
        """
        Creamos la API simulada en estado 'ok' (todo funciona)
        para cada test. En un proyecto real, apuntaría al
        servidor del ambiente de QA/staging.
        """
        # Creamos la API con estado 'ok' para simular que todo funciona
        self.api = SimuladorAPIBiblioteca(estado='ok')

    # ── SMOKE TEST 1: ¿El servidor responde? ─────────────────────
    def test_smoke_01_servidor_responde(self):
        """
        SMOKE: Verificación más básica posible.
        Si el servidor no responde a /health, ningún otro test
        tiene sentido — el build está completamente roto.
        """
        # Hacemos la petición más simple posible
        respuesta = self.api.get('/api/health')
        
        # El código 200 significa 'OK' en HTTP
        assert respuesta.status_code == 200, (
            f"El servidor devolvió código {respuesta.status_code} "
            "en vez de 200. El servidor no está operativo."
        )
        
        # El campo 'status' debe ser 'ok'
        assert respuesta.json()['status'] == 'ok', (
            f"El estado del servidor es '{respuesta.json()['status']}' en vez de 'ok'"
        )

    # ── SMOKE TEST 2: ¿La base de datos está conectada? ──────────
    def test_smoke_02_base_de_datos_conectada(self):
        """
        SMOKE: Sin base de datos, la aplicación no funciona.
        Este es el segundo test más crítico después del servidor.
        """
        respuesta = self.api.get('/api/health/db')
        
        # La BD debe estar conectada
        assert respuesta.status_code == 200
        assert respuesta.json()['db_status'] == 'connected', (
            f"La BD reporta: {respuesta.json().get('db_status', 'desconocido')}. "
            "Sin BD, ninguna funcionalidad del sistema funciona."
        )

    # ── SMOKE TEST 3: ¿El login funciona? ────────────────────────
    def test_smoke_03_login_exitoso(self):
        """
        SMOKE: La autenticación es la puerta de entrada
        a todas las funcionalidades. Si el login falla,
        ningún usuario puede usar el sistema.
        """
        # Usamos credenciales de prueba preconfiguradas en el sistema
        respuesta = self.api.post('/api/auth/login', data={
            'email': 'test@uni.edu.co',
            'password': 'Test123!'
        })
        
        # El login debe retornar 200 (OK)
        assert respuesta.status_code == 200, (
            f"El login retornó código {respuesta.status_code}. "
            "La autenticación no está funcionando."
        )
        
        # La respuesta debe incluir el token de acceso
        assert 'access_token' in respuesta.json(), (
            "La respuesta del login no incluye 'access_token'. "
            "Los usuarios no podrán autenticarse."
        )

    # ── SMOKE TEST 4: ¿El catálogo de libros responde? ───────────
    def test_smoke_04_catalogo_libros_accesible(self):
        """
        SMOKE: La funcionalidad core de la biblioteca.
        Si el catálogo no responde, los usuarios no pueden
        buscar ni reservar libros.
        """
        respuesta = self.api.get('/api/libros')
        
        # Puede ser 200 (con libros) o 401 (requiere auth) — ambos son válidos
        # Lo que NO puede ser: 500 (error del servidor) o no responder
        assert respuesta.status_code in [200, 401], (
            f"El catálogo devolvió código {respuesta.status_code}. "
            "Solo son aceptables 200 (OK) o 401 (requiere autenticación)."
        )

    # ── SMOKE TEST 5: ¿El módulo de multas está desplegado? ──────
    def test_smoke_05_modulo_multas_desplegado(self):
        """
        SMOKE: Verificamos que el nuevo módulo de multas
        fue correctamente desplegado en este build.
        No hacemos pruebas profundas — solo verificamos que existe.
        """
        respuesta = self.api.get('/api/multas/2024-001')
        
        # El endpoint debe existir y responder (no 404 ni 500)
        # 200 = OK, 401 = requiere auth, 403 = sin permisos (todos válidos)
        assert respuesta.status_code != 404, (
            "El endpoint /api/multas/{codigo} no existe (404). "
            "El módulo de multas no fue desplegado."
        )
        assert respuesta.status_code != 500, (
            "El módulo de multas devolvió un error 500. "
            "Hay un error en el servidor."
        )

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 3.3 — Demostración: Cómo se comporta el Smoke Test
#              cuando el sistema falla
# ═══════════════════════════════════════════════════════════════════

%%ipytest -v

class TestSmokeFallaConBDCaida:
    """
    Demostración: Los smoke tests con la BD caída.
    
    Esto muestra cómo los smoke tests detectan problemas
    de infraestructura, no solo de código.
    """

    def setup_method(self):
        # Creamos la API simulando que la BD está CAÍDA
        self.api_con_db_caida = SimuladorAPIBiblioteca(estado='db_down')

    def test_smoke_detecta_bd_desconectada(self):
        """
        Este test PASA porque está diseñado para detectar
        que la BD está caída — y efectivamente lo detecta.
        """
        respuesta = self.api_con_db_caida.get('/api/health/db')
        
        # Cuando la BD está caída, debe responder con status que indique el problema
        # 503 = Service Unavailable (BD no disponible)
        assert respuesta.status_code == 503, (
            "Cuando la BD está caída, el endpoint /health/db "
            "debe responder con 503 Service Unavailable"
        )
        
        # El estado de la BD debe ser 'disconnected'
        assert respuesta.json()['db_status'] == 'disconnected'
        
        print("\n💡 SMOKE DETECTÓ: La base de datos está desconectada.")
        print("   Acción requerida: Rechazar el build y notificar al equipo de infraestructura.")

    def test_smoke_catalogo_no_disponible_con_bd_caida(self):
        """
        Con la BD caída, el catálogo de libros tampoco funciona.
        El smoke test detecta el fallo en cascada.
        """
        respuesta = self.api_con_db_caida.get('/api/libros')
        
        # Con BD caída, el catálogo devuelve 503
        assert respuesta.status_code == 503, (
            f"Con BD caída, el catálogo debe devolver 503, "
            f"pero devolvió {respuesta.status_code}"
        )

---
# ✅ SECCIÓN 4 — ACCEPTANCE TESTING (Pruebas de Aceptación)

## ¿Qué es Acceptance Testing?

Las **pruebas de aceptación** verifican que el sistema cumple los **criterios de aceptación**  
definidos por el cliente o usuario final. Son la **última barrera** antes de entregar el software.

### La diferencia clave:
```
QA dice:    "La función calcular_multa() retorna el valor correcto" → TÉCNICO
Cliente dice: "Cuando devuelvo el libro 3 días tarde, me cobran $1500" → NEGOCIO
```
Las pruebas de aceptación hablan el lenguaje del **cliente**, no del desarrollador.

### BDD y Gherkin:
**Behavior Driven Development (BDD)** usa el formato **DADO/CUANDO/ENTONCES** para escribir  
criterios de aceptación en lenguaje natural que también pueden ejecutarse como tests.

```gherkin
Escenario: Cliente devuelve libro tarde
  DADO   que Carlos tiene el libro prestado desde hace 17 días
  CUANDO devuelve el libro hoy
  ENTONCES el sistema calcula una multa de $1500 COP
  Y        Carlos queda bloqueado para nuevos préstamos
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 4.1 — Framework de BDD implementado desde cero
#
#  Para evitar instalar Behave y configurar archivos .feature,
#  implementamos un mini-framework de BDD directamente en Python.
#  Esto muestra la ESENCIA de cómo funcionan Behave y Cucumber.
# ═══════════════════════════════════════════════════════════════════

class ContextoBDD:
    """
    El 'contexto' es el objeto que se pasa entre los pasos
    DADO, CUANDO y ENTONCES de un escenario BDD.
    
    Sirve para compartir datos entre pasos:
    - El paso DADO crea los datos
    - El paso CUANDO ejecuta la acción y guarda el resultado
    - El paso ENTONCES verifica el resultado
    """
    pass  # el contexto es simplemente un objeto donde guardar variables


class EscenarioBDD:
    """
    Simula un escenario de Gherkin (Feature/Scenario) en Python puro.
    
    Permite escribir tests en formato:
        escenario.dado("condición").cuando("acción").entonces("verificación")
    """
    
    def __init__(self, nombre: str):
        """
        Args:
            nombre: descripción del escenario (aparece en los reportes)
        """
        self.nombre = nombre
        # Creamos un contexto vacío que se irá llenando con los datos del escenario
        self.ctx = ContextoBDD()
        # Lista de pasos ejecutados (para el reporte)
        self._pasos_ejecutados = []
        print(f"\n📋 Escenario: {nombre}")
    
    def dado(self, descripcion: str, accion=None):
        """
        Corresponde a 'Given' en Gherkin: prepara el contexto inicial.
        
        Args:
            descripcion: descripción legible del paso
            accion:      función opcional que configura el contexto
        """
        print(f"   DADO    {descripcion}")
        if accion:  # si se pasó una función, la ejecutamos
            accion(self.ctx)
        self._pasos_ejecutados.append(('DADO', descripcion))
        return self  # retornamos self para poder encadenar .cuando()
    
    def cuando(self, descripcion: str, accion=None):
        """
        Corresponde a 'When' en Gherkin: ejecuta la acción del escenario.
        """
        print(f"   CUANDO  {descripcion}")
        if accion:
            accion(self.ctx)
        self._pasos_ejecutados.append(('CUANDO', descripcion))
        return self
    
    def entonces(self, descripcion: str, verificacion=None):
        """
        Corresponde a 'Then' en Gherkin: verifica el resultado esperado.
        """
        print(f"   ENTONCES {descripcion}")
        if verificacion:
            verificacion(self.ctx)
        self._pasos_ejecutados.append(('ENTONCES', descripcion))
        return self
    
    def y(self, descripcion: str, accion=None):
        """
        Corresponde a 'And' en Gherkin: agrega un paso adicional.
        Se puede usar después de DADO, CUANDO o ENTONCES.
        """
        print(f"   Y       {descripcion}")
        if accion:
            accion(self.ctx)
        self._pasos_ejecutados.append(('Y', descripcion))
        return self
    
    def verificar(self) -> bool:
        """Ejecuta todos los pasos y retorna True si todos pasaron."""
        # Si llegamos aquí sin excepciones, todos los pasos pasaron
        print(f"   → ✅ ESCENARIO PASA\n")
        return True


print("✅ Framework BDD creado.")
print("   Uso: EscenarioBDD('nombre').dado(...).cuando(...).entonces(...).verificar()")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 4.2 — Criterios de Aceptación escritos en BDD
#
#  Estos escenarios corresponden a los criterios que el cliente
#  (el Director de la Biblioteca) aprobó antes del desarrollo.
#  Son el contrato entre el cliente y el equipo de desarrollo.
# ═══════════════════════════════════════════════════════════════════

print("="*65)
print(" ACCEPTANCE TESTS — Feature: Sistema de Multas de Biblioteca")
print("="*65)
print(" Cliente: Dirección de la Biblioteca Universitaria")
print(" Validado por: Director Académico")
print("="*65)

# ═══════════════════════════════════════════════════════════════════
# CRITERIO DE ACEPTACIÓN 1 (AC-001)
# Redacción del cliente: "Cuando un estudiante devuelva un libro
# con N días de retraso, el sistema debe cobrar N × $500 pesos"
# ═══════════════════════════════════════════════════════════════════

def ac001_escenario_multa_por_tres_dias():
    """
    AC-001: Cálculo correcto de multa por retraso.
    
    Historia de usuario:
    COMO Director de la Biblioteca
    QUIERO que el sistema calcule multas automáticamente
    PARA incentivar la devolución oportuna de los libros
    """
    
    escenario = EscenarioBDD("AC-001: Multa de $1500 por 3 días de retraso")
    
    # ── DADO: preparamos el contexto ────────────────────────────
    def preparar_prestamo_vencido(ctx):
        """
        Configuramos el contexto inicial del escenario:
        - Un estudiante tiene un libro prestado
        - La fecha límite ya pasó hace 3 días
        """
        ctx.estudiante = Estudiante("2024-003", "Carlos Pérez")
        ctx.libro = Libro(1, "Clean Code", "Robert C. Martin", disponible=False)
        
        # Calculamos una fecha límite que ya venció hace 3 días
        hoy = datetime.date.today()
        ctx.fecha_limite = hoy - datetime.timedelta(days=3)  # 3 días atrás
        
        ctx.dias_retraso = 3  # guardamos para usar en los pasos siguientes
    
    # ── CUANDO: ejecutamos la acción ────────────────────────────
    def devolver_libro_tarde(ctx):
        """
        El estudiante devuelve el libro hoy (con 3 días de retraso).
        Calculamos la multa.
        """
        # Calculamos la multa por los días de retraso
        ctx.multa_calculada = calcular_multa_por_dias(ctx.dias_retraso)
        
        # Aplicamos la multa al estudiante
        aplicar_multa_a_estudiante(ctx.estudiante, ctx.multa_calculada)
    
    # ── ENTONCES: verificamos los resultados ────────────────────
    def verificar_multa_correcta(ctx):
        """
        La multa debe ser exactamente 3 × $500 = $1500 COP.
        """
        multa_esperada = 1500.0  # 3 días × $500
        assert ctx.multa_calculada == multa_esperada, (
            f"AC-001 FALLA: Se esperaba multa de ${multa_esperada} COP "
            f"pero se calculó ${ctx.multa_calculada} COP"
        )
    
    def verificar_desglose_correcto(ctx):
        """
        El desglose debe ser: 3 días × $500 = $1500.
        Verificamos que la fórmula sea correcta.
        """
        multa_por_formula = ctx.dias_retraso * TARIFA_DIARIA_MULTA
        assert ctx.multa_calculada == multa_por_formula
    
    # ── Ejecutamos el escenario completo ────────────────────────
    escenario \
        .dado("Carlos Pérez tiene el libro 'Clean Code' prestado",
               preparar_prestamo_vencido) \
        .y("la fecha límite de devolución fue hace 3 días") \
        .cuando("Carlos devuelve el libro hoy",
                 devolver_libro_tarde) \
        .entonces("el sistema muestra una multa de $1500 COP",
                   verificar_multa_correcta) \
        .y("el desglose es: 3 días × $500 = $1500",
            verificar_desglose_correcto) \
        .verificar()


# Ejecutamos el primer criterio de aceptación
ac001_escenario_multa_por_tres_dias()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 4.3 — Criterios de Aceptación 2 y 3
# ═══════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════
# CRITERIO DE ACEPTACIÓN 2 (AC-002)
# "Cuando un estudiante tenga multa pendiente, NO podrá hacer
#  nuevos préstamos hasta que pague la deuda completa"
# ═══════════════════════════════════════════════════════════════════

def ac002_escenario_bloqueo_por_multa():
    """
    AC-002: Bloqueo automático por multa pendiente.
    """
    escenario = EscenarioBDD("AC-002: Estudiante con multa no puede reservar libros")
    
    def preparar_estudiante_con_multa(ctx):
        # Creamos un libro disponible para intentar prestarlo
        ctx.libro_nuevo = Libro(2, "Refactoring", "Martin Fowler", disponible=True)
        
        # Creamos un estudiante que tiene multa pendiente
        ctx.estudiante = Estudiante(
            codigo="2024-MULT",
            nombre="Luis Torres"
        )
        # Le aplicamos una multa para bloquearlo
        aplicar_multa_a_estudiante(ctx.estudiante, 2000.0)  # $2000 de multa
    
    def intentar_nuevo_prestamo(ctx):
        """
        Luis intenta reservar un libro a pesar de tener multa.
        Guardamos si hubo error.
        """
        try:
            registrar_prestamo(ctx.libro_nuevo, ctx.estudiante, [])
            ctx.error_generado = None  # no hubo error
            ctx.prestamo_realizado = True
        except ValueError as e:
            # Guardamos el error para verificarlo en 'entonces'
            ctx.error_generado = str(e)
            ctx.prestamo_realizado = False
    
    def verificar_prestamo_rechazado(ctx):
        """
        El préstamo NO debe haberse realizado.
        """
        assert ctx.prestamo_realizado == False, (
            "AC-002 FALLA: El sistema permitió un préstamo a un "
            "estudiante con multa pendiente."
        )
    
    def verificar_mensaje_de_bloqueo(ctx):
        """
        El sistema debe mostrar un mensaje claro de bloqueo.
        """
        assert ctx.error_generado is not None, "Debe haber un mensaje de error"
        # El mensaje debe ser comprensible para el usuario
        assert "multa" in ctx.error_generado.lower(), (
            f"El mensaje de error debe mencionar la multa: '{ctx.error_generado}'"
        )
    
    escenario \
        .dado("Luis Torres tiene una multa pendiente de $2000 COP",
               preparar_estudiante_con_multa) \
        .y("el libro 'Refactoring' tiene cupo disponible") \
        .cuando("Luis intenta reservar el libro 'Refactoring'",
                 intentar_nuevo_prestamo) \
        .entonces("el sistema rechaza la reserva",
                   verificar_prestamo_rechazado) \
        .y("muestra el mensaje indicando la multa pendiente",
            verificar_mensaje_de_bloqueo) \
        .verificar()


# ═══════════════════════════════════════════════════════════════════
# CRITERIO DE ACEPTACIÓN 3 (AC-003)
# "Cuando un estudiante pague su multa completa, su cuenta
#  se debe reactivar automáticamente y podrá hacer préstamos"
# ═══════════════════════════════════════════════════════════════════

def ac003_escenario_reactivacion_tras_pago():
    """
    AC-003: Reactivación automática de cuenta al pagar multa.
    """
    escenario = EscenarioBDD("AC-003: Cuenta se reactiva al pagar la multa completa")
    
    def preparar_estudiante_bloqueado(ctx):
        ctx.estudiante = Estudiante(
            codigo="2024-PAY",
            nombre="María Rodríguez",
            activo=False,
            multa_pendiente=1500.0  # debe $1500
        )
        ctx.libro = Libro(3, "The Pragmatic Programmer", "Hunt & Thomas")
    
    def pagar_multa_completa(ctx):
        # María paga exactamente lo que debe
        ctx.saldo_restante = pagar_multa(ctx.estudiante, monto_pago=1500.0)
    
    def verificar_cuenta_activa(ctx):
        assert ctx.saldo_restante == 0.0, (
            f"Después de pagar $1500, el saldo debe ser $0 "
            f"pero quedó ${ctx.saldo_restante}"
        )
        assert ctx.estudiante.activo == True, (
            "AC-003 FALLA: Después de pagar la multa completa, "
            "la cuenta debe quedar ACTIVA"
        )
    
    def verificar_puede_hacer_prestamos(ctx):
        # Verificamos que ahora sí puede hacer un préstamo
        try:
            prestamo = registrar_prestamo(ctx.libro, ctx.estudiante, [])
            assert prestamo is not None, "El préstamo debe haberse creado"
        except ValueError as e:
            # Si lanza error, el criterio de aceptación FALLA
            assert False, (
                f"AC-003 FALLA: Después de pagar la multa, el sistema "
                f"sigue bloqueando el préstamo: {e}"
            )
    
    escenario \
        .dado("María Rodríguez tiene cuenta bloqueada por multa de $1500 COP",
               preparar_estudiante_bloqueado) \
        .cuando("María paga la multa completa de $1500 COP en caja",
                 pagar_multa_completa) \
        .entonces("su cuenta queda con saldo pendiente de $0",
                   verificar_cuenta_activa) \
        .y("María puede realizar nuevos préstamos normalmente",
            verificar_puede_hacer_prestamos) \
        .verificar()


# Ejecutamos los criterios de aceptación
ac002_escenario_bloqueo_por_multa()
ac003_escenario_reactivacion_tras_pago()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 4.4 — Acceptance Tests como unittest (formato alternativo)
#
#  Aquí mostramos la misma lógica de acceptance testing pero
#  usando el formato clásico de pytest, más parecido a lo que
#  se usaría con Behave en un proyecto real.
# ═══════════════════════════════════════════════════════════════════

%%ipytest -v

class TestAcceptanceSistemaMultas:
    """
    Suite formal de pruebas de aceptación para el sistema de multas.
    
    Cada test representa un criterio de aceptación (AC)
    que fue definido y aprobado por el cliente.
    
    Los nombres siguen el patrón:
    test_ac{numero}_{descripcion_del_criterio}
    """

    # ── AC-001 ───────────────────────────────────────────────────
    def test_ac001_multa_correcta_por_dias_de_retraso(self):
        """
        AC-001: Criterio de aceptación del Director de Biblioteca.
        
        'Al devolver un libro con N días de retraso,
         el sistema debe calcular una multa de N × $500 COP'
        
        Aprobado por: Director Académico — 15/03/2024
        """
        # Tabla de casos (casos de prueba del criterio de aceptación)
        # Cada fila es: (días de retraso, multa esperada en COP)
        casos = [
            (0,  0.0),     # devuelto a tiempo → sin multa
            (1,  500.0),   # 1 día tarde → $500
            (3,  1500.0),  # 3 días tarde → $1500 (caso más común)
            (7,  3500.0),  # 1 semana tarde → $3500
            (14, 7000.0),  # 2 semanas tarde → $7000
            (30, 15000.0), # 1 mes tarde → $15000
        ]
        
        for dias, multa_esperada in casos:
            # ACT: calculamos la multa
            multa_calculada = calcular_multa_por_dias(dias)
            
            # ASSERT: verificamos el criterio del cliente
            assert multa_calculada == multa_esperada, (
                f"AC-001 FALLA para {dias} días de retraso: "
                f"Se esperaba ${multa_esperada} COP pero se calculó ${multa_calculada} COP. "
                f"Fórmula esperada: {dias} días × $500 COP/día"
            )

    # ── AC-002 ───────────────────────────────────────────────────
    def test_ac002_estudiante_sin_multa_puede_prestar(self):
        """
        AC-002: Un estudiante sin multa pendiente puede
        realizar préstamos normalmente.
        """
        # ARRANGE: estudiante activo sin multas
        estudiante = Estudiante("2024-A001", "Juan Sin Multa", activo=True, multa_pendiente=0.0)
        libro = Libro(10, "Domain Driven Design", "Eric Evans", disponible=True)
        
        # ACT + ASSERT: el préstamo no debe lanzar excepciones
        try:
            prestamo = registrar_prestamo(libro, estudiante, [])
            assert prestamo is not None
            assert isinstance(prestamo, Prestamo)
        except ValueError as e:
            pytest.fail(
                f"AC-002 FALLA: Un estudiante sin multa no debería "
                f"ser bloqueado para hacer préstamos. Error: {e}"
            )

    # ── AC-003 ───────────────────────────────────────────────────
    def test_ac003_mensaje_claro_cuando_sin_multa(self):
        """
        AC-003: Cuando el estudiante no tiene multa,
        al consultar multas debe ver 'Sin multa pendiente'.
        
        Este criterio define el MENSAJE que ve el usuario,
        no solo el valor numérico.
        """
        # ARRANGE: estudiante sin multa
        estudiante = Estudiante("2024-CLEAN", "Sin Deudas")
        
        # ACT: generamos el mensaje de estado de multa
        def generar_mensaje_multa(est: Estudiante) -> str:
            """
            Función que genera el mensaje visible al usuario.
            En la app real, esto sería un endpoint de la API.
            """
            if est.multa_pendiente == 0.0:
                return "Sin multa pendiente"  # mensaje para el usuario
            return f"Multa pendiente: ${est.multa_pendiente:,.0f} COP"
        
        mensaje = generar_mensaje_multa(estudiante)
        
        # ASSERT: verificamos que el mensaje sea el correcto
        assert "Sin multa pendiente" in mensaje, (
            f"AC-003 FALLA: Con multa $0, el mensaje debe decir "
            f"'Sin multa pendiente' pero dice: '{mensaje}'"
        )

    # ── AC-004 ───────────────────────────────────────────────────
    def test_ac004_flujo_completo_prestamo_retraso_pago_reactivacion(self):
        """
        AC-004: Flujo completo de usuario.
        
        Este criterio verifica el recorrido completo de un estudiante:
        Préstamo → Retraso → Multa → Bloqueo → Pago → Reactivación
        
        Es el criterio más importante porque valida el flujo
        de negocio de principio a fin.
        """
        # PASO 1: Estudiante activo hace un préstamo
        estudiante = Estudiante("2024-FLUJO", "Estudiante Flujo")
        libro = Libro(20, "Working Effectively with Legacy Code", "Michael Feathers")
        
        prestamo = registrar_prestamo(libro, estudiante, [])
        assert prestamo is not None
        assert libro.disponible == False  # el libro ya no está disponible
        
        # PASO 2: El estudiante devuelve tarde (5 días de retraso)
        # Simulamos que devuelve 5 días después de la fecha límite
        fecha_devolucion_tarde = prestamo.fecha_limite + datetime.timedelta(days=5)
        multa = registrar_devolucion(prestamo, libro, fecha_devolucion=fecha_devolucion_tarde)
        
        assert multa == 2500.0  # 5 días × $500 = $2500
        assert libro.disponible == True   # el libro volvió a estar disponible
        assert prestamo.devuelto == True  # el préstamo está cerrado
        
        # PASO 3: Se aplica la multa → estudiante queda bloqueado
        aplicar_multa_a_estudiante(estudiante, multa)
        
        assert estudiante.activo == False          # bloqueado
        assert estudiante.multa_pendiente == 2500.0  # debe $2500
        
        # PASO 4: El estudiante intenta hacer otro préstamo → debe ser rechazado
        libro2 = Libro(21, "Clean Architecture", "Robert C. Martin")
        with pytest.raises(ValueError):
            registrar_prestamo(libro2, estudiante, [])
        
        # PASO 5: El estudiante paga la multa completa → debe reactivarse
        saldo = pagar_multa(estudiante, monto_pago=2500.0)
        
        assert saldo == 0.0
        assert estudiante.activo == True  # reactivado
        
        # PASO 6: Ahora sí puede hacer un nuevo préstamo
        prestamo2 = registrar_prestamo(libro2, estudiante, [])
        assert prestamo2 is not None  # el préstamo fue exitoso
        
        print("\n✅ AC-004 PASA: Flujo completo Préstamo→Retraso→Multa→Pago→Reactivación")

---
# 🔗 SECCIÓN 5 — Integración: Los 3 Tipos en un Pipeline

En un proyecto real, los tres tipos de prueba se ejecutan en **secuencia** como parte de un pipeline de CI/CD.

```
Desarrollador hace commit
        ↓
  [1] SMOKE TESTS (5-10 min)
       ¿El sistema arranca?
       FALLA → Rechazar build inmediatamente
       PASA  ↓
  [2] REGRESSION TESTS (30-60 min)
       ¿Nada se rompió?
       FALLA → Bloquear el merge
       PASA  ↓
  [3] ACCEPTANCE TESTS (con cliente)
       ¿Cumple los criterios del cliente?
       FALLA → No hay release
       PASA  → 🚀 Release a producción
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 5.1 — Pipeline integrado de los 3 tipos de prueba
# ═══════════════════════════════════════════════════════════════════

def ejecutar_pipeline_completo(api: SimuladorAPIBiblioteca, verbose: bool = True) -> dict:
    """
    Simula la ejecución del pipeline completo de pruebas.
    
    Este pipeline representa lo que ocurriría en GitHub Actions,
    Jenkins, GitLab CI u otro sistema de CI/CD.
    
    Args:
        api:     la API del sistema a probar
        verbose: si True, muestra los detalles de cada paso
    
    Returns:
        Diccionario con los resultados: {'smoke': bool, 'regression': bool, 'acceptance': bool}
    """
    resultados = {
        'smoke':      False,
        'regression': False,
        'acceptance': False,
        'puede_hacer_release': False
    }
    
    separador = "═" * 60
    
    # ════════════════════════════════════════════════════════════
    # FASE 1: SMOKE TESTS
    # ════════════════════════════════════════════════════════════
    if verbose:
        print(f"\n{separador}")
        print("  FASE 1: SMOKE TESTS")
        print(f"{separador}")
    
    # Definimos los checks del smoke test
    smoke_checks = [
        ("Servidor disponible",    lambda: api.get('/api/health').status_code == 200),
        ("Base de datos conectada",lambda: api.get('/api/health/db').status_code == 200),
        ("Login operativo",        lambda: api.post('/api/auth/login', {'email':'test@uni.edu.co','password':'Test123!'}).status_code == 200),
        ("Catálogo accesible",     lambda: api.get('/api/libros').status_code in [200, 401]),
        ("Módulo multas activo",   lambda: api.get('/api/multas/2024-001').status_code != 500),
    ]
    
    smoke_fallidos = []  # lista de checks que fallaron
    
    for nombre, check in smoke_checks:
        try:
            resultado = check()  # ejecutamos el check
            icono = "✅" if resultado else "❌"
            if verbose:
                print(f"  {icono} {nombre}")
            if not resultado:
                smoke_fallidos.append(nombre)  # registramos el fallo
        except Exception as e:
            # Si ocurre una excepción (ej: servidor caído), también es un fallo
            if verbose:
                print(f"  ❌ {nombre} → ERROR: {e}")
            smoke_fallidos.append(nombre)
    
    # Evaluamos el resultado del smoke test
    resultados['smoke'] = len(smoke_fallidos) == 0
    
    if verbose:
        if resultados['smoke']:
            print("\n  ✅ SMOKE: TODOS LOS CHECKS PASARON")
        else:
            print(f"\n  ❌ SMOKE: FALLARON {len(smoke_fallidos)} CHECKS → BUILD RECHAZADO")
            print(f"     Checks fallidos: {', '.join(smoke_fallidos)}")
    
    # PRINCIPIO FAIL-FAST: si falla el smoke, no ejecutamos más
    if not resultados['smoke']:
        if verbose:
            print("\n  🛑 Pipeline detenido en Smoke Tests.")
            print("  Acción: Notificar al equipo de desarrollo.")
        return resultados
    
    # ════════════════════════════════════════════════════════════
    # FASE 2: REGRESSION TESTS (solo si el smoke pasó)
    # ════════════════════════════════════════════════════════════
    if verbose:
        print(f"\n{separador}")
        print("  FASE 2: REGRESSION TESTS")
        print(f"{separador}")
    
    # Ejecutamos los casos de regresión más críticos
    regression_checks = [
        ("Multa $0 por devolución a tiempo", lambda: calcular_multa_por_dias(0) == 0.0),
        ("Fórmula de multa correcta",        lambda: calcular_multa_por_dias(3) == 1500.0),
        ("Tipo de retorno es numérico",       lambda: isinstance(calcular_multa_por_dias(5), (int, float))),
        ("Libro queda no disponible",         lambda: _check_libro_no_disponible()),
        ("Pago reactiva la cuenta",           lambda: _check_pago_reactiva()),
    ]
    
    # Funciones auxiliares para los checks más complejos
    def _check_libro_no_disponible() -> bool:
        libro = Libro(99, "Test", "Autor")
        estudiante = Estudiante("TEST", "Test")
        registrar_prestamo(libro, estudiante, [])
        return libro.disponible == False
    
    def _check_pago_reactiva() -> bool:
        est = Estudiante("PAY", "Pago", activo=False, multa_pendiente=1000.0)
        pagar_multa(est, 1000.0)
        return est.activo == True
    
    regression_fallidos = []
    
    for nombre, check in regression_checks:
        try:
            resultado = check()
            icono = "✅" if resultado else "❌"
            if verbose:
                print(f"  {icono} {nombre}")
            if not resultado:
                regression_fallidos.append(nombre)
        except Exception as e:
            if verbose:
                print(f"  ❌ {nombre} → EXCEPCIÓN: {e}")
            regression_fallidos.append(nombre)
    
    resultados['regression'] = len(regression_fallidos) == 0
    
    if verbose:
        if resultados['regression']:
            print("\n  ✅ REGRESSION: TODOS LOS CHECKS PASARON")
        else:
            print(f"\n  ❌ REGRESSION: FALLARON {len(regression_fallidos)} CHECKS → BLOQUEAR MERGE")
    
    if not resultados['regression']:
        if verbose:
            print("\n  🛑 Pipeline detenido en Regression Tests.")
        return resultados
    
    # ════════════════════════════════════════════════════════════
    # FASE 3: ACCEPTANCE TESTS (solo si regression pasó)
    # ════════════════════════════════════════════════════════════
    if verbose:
        print(f"\n{separador}")
        print("  FASE 3: ACCEPTANCE TESTS")
        print(f"{separador}")
    
    acceptance_checks = [
        ("AC-001: Cálculo correcto de multas",  lambda: calcular_multa_por_dias(3) == 1500.0 and calcular_multa_por_dias(5) == 2500.0),
        ("AC-002: Bloqueo por multa pendiente",  lambda: _check_bloqueo_multa()),
        ("AC-003: Reactivación al pagar",         lambda: _check_reactivacion()),
    ]
    
    def _check_bloqueo_multa() -> bool:
        est = Estudiante("BLQ", "Bloqueado", activo=False, multa_pendiente=500.0)
        libro = Libro(98, "T", "A")
        try:
            registrar_prestamo(libro, est, [])
            return False  # no debería llegar aquí
        except ValueError:
            return True   # debe lanzar error
    
    def _check_reactivacion() -> bool:
        est = Estudiante("REA", "Reactiva", activo=False, multa_pendiente=500.0)
        pagar_multa(est, 500.0)
        return est.activo == True
    
    acceptance_fallidos = []
    
    for nombre, check in acceptance_checks:
        try:
            resultado = check()
            icono = "✅" if resultado else "❌"
            if verbose:
                print(f"  {icono} {nombre}")
            if not resultado:
                acceptance_fallidos.append(nombre)
        except Exception as e:
            if verbose:
                print(f"  ❌ {nombre} → EXCEPCIÓN: {e}")
            acceptance_fallidos.append(nombre)
    
    resultados['acceptance'] = len(acceptance_fallidos) == 0
    resultados['puede_hacer_release'] = all([
        resultados['smoke'],
        resultados['regression'],
        resultados['acceptance']
    ])
    
    # ════════════════════════════════════════════════════════════
    # RESULTADO FINAL
    # ════════════════════════════════════════════════════════════
    if verbose:
        print(f"\n{separador}")
        print("  RESULTADO FINAL DEL PIPELINE")
        print(f"{separador}")
        print(f"  Smoke Tests:      {'✅ PASA' if resultados['smoke'] else '❌ FALLA'}")
        print(f"  Regression Tests: {'✅ PASA' if resultados['regression'] else '❌ FALLA'}")
        print(f"  Acceptance Tests: {'✅ PASA' if resultados['acceptance'] else '❌ FALLA'}")
        print(f"{separador}")
        if resultados['puede_hacer_release']:
            print("  🚀 TODOS LOS TESTS PASAN → LISTO PARA RELEASE A PRODUCCIÓN")
        else:
            print("  🛑 HAY TESTS FALLANDO → NO SE PUEDE HACER RELEASE")
        print(f"{separador}")
    
    return resultados


# ── Ejecutamos el pipeline con el sistema en estado normal ───────
print("ESCENARIO 1: Sistema funcionando correctamente")
api_ok = SimuladorAPIBiblioteca(estado='ok')
resultados = ejecutar_pipeline_completo(api_ok)

print("\n" + "─"*60)
print("ESCENARIO 2: Sistema con base de datos caída")
api_db_down = SimuladorAPIBiblioteca(estado='db_down')
resultados_db_down = ejecutar_pipeline_completo(api_db_down)

---
# 📊 SECCIÓN 6 — Resumen Final y Comparativa

## Los 3 tipos en una sola tabla:

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 6.1 — Tabla resumen de los 3 tipos de prueba
# ═══════════════════════════════════════════════════════════════════

# Usamos el módulo tabulate para generar una tabla bien formateada
# Si no está instalado, lo instalamos
try:
    from tabulate import tabulate
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'tabulate', '--quiet'])
    from tabulate import tabulate

# Definimos los datos de la tabla comparativa
datos_tabla = [
    # [Dimensión, Regression, Smoke, Acceptance]
    ["¿Qué verifica?",
     "Que lo que ya\nfuncionaba\nsiga funcionando",
     "Que lo básico\ndel sistema\nesté operativo",
     "Que cumpla los\ncriterios del\ncliente"],
    
    ["¿Cuándo?",
     "Después de CADA\ncambio al código",
     "Al recibir\nun nuevo build",
     "Antes del\nrelease final"],
    
    ["¿Quién?",
     "QA Automation\nEquipo Dev",
     "CI/CD\n(automático)",
     "Cliente\nProduct Owner"],
    
    ["¿Cuánto tarda?",
     "30 min - 2 horas",
     "5 - 15 minutos",
     "Horas - Días"],
    
    ["¿Cuántos tests?",
     "Muchos\n(cobertura alta)",
     "Pocos\n(5-15% críticos)",
     "Los necesarios\n(por criterio)"],
    
    ["Si falla...",
     "Bloquear el\nmerge/PR",
     "Rechazar el\nbuild completo",
     "No hay release\nhasta corregir"],
    
    ["Automatizable",
     "✅ Siempre",
     "✅ Siempre",
     "⚠️ Parcialmente"],
    
    ["Herramientas Python",
     "pytest, unittest",
     "pytest -m smoke",
     "behave, pytest"],
    
    ["Ejemplo en el curso",
     "TestMultasRegression",
     "TestBibliotecaSmoke",
     "ac001, ac002, ac003"],
]

# Encabezados de la tabla
encabezados = ["Dimensión", "🔁 Regression\nTesting", "💨 Smoke\nTesting", "✅ Acceptance\nTesting"]

# Imprimimos la tabla
print("\n" + "="*75)
print("  COMPARATIVA: REGRESSION vs SMOKE vs ACCEPTANCE TESTING")
print("="*75)
print(tabulate(
    datos_tabla,
    headers=encabezados,
    tablefmt="grid",      # formato con líneas
    stralign="center"     # texto centrado
))
print("="*75)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 6.2 — Resumen ejecutivo de lo aprendido
# ═══════════════════════════════════════════════════════════════════

print("""
╔══════════════════════════════════════════════════════════════════╗
║          RESUMEN EJECUTIVO — SESIONES 6 Y 7                     ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🔁 REGRESSION TESTING                                          ║
║     • Protege el comportamiento ya validado                      ║
║     • Detecta cuando un cambio rompe algo que funcionaba         ║
║     • Marca los tests con @pytest.mark.regression                ║
║     • Siempre incluye tests para bugs históricos                 ║
║                                                                  ║
║  💨 SMOKE TESTING                                               ║
║     • Subset mínimo de tests críticos (5-15%)                    ║
║     • Responde: ¿el sistema arranca y responde?                  ║
║     • Máximo 15 minutos de ejecución                             ║
║     • Si falla → rechazar el build sin más pruebas               ║
║                                                                  ║
║  ✅ ACCEPTANCE TESTING                                          ║
║     • Valida que el sistema cumple los requisitos del cliente    ║
║     • Escrito en lenguaje de negocio (BDD/Gherkin)               ║
║     • El cliente/PO es quien aprueba los resultados              ║
║     • Patrón: DADO/CUANDO/ENTONCES                               ║
║                                                                  ║
║  🔄 SECUENCIA EN CI/CD:                                         ║
║     Smoke → Regression → Acceptance → Release                   ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  Mg. Sergio Alejandro Burbano Mena                               ║
║  Pruebas de Software — 7mo Semestre                              ║
║  Universitaria de Colombia                                       ║
╚══════════════════════════════════════════════════════════════════╝
""")